# Follow the Leader: Enhancing Systematic Trend-Following Using Network Momentum

**Authors:** Linze Li, William Ferreira
**Published:** 2025-01-13
**ArXiv:** [https://arxiv.org/abs/2501.07135](https://arxiv.org/abs/2501.07135)

**Abstract:**
We present a systematic, trend-following strategy, applied to commodity futures markets, that combines univariate trend indicators with cross-sectional trend indicators that capture so-called *momentum spillover*, which can occur when there is a lead-lag relationship between the trending behaviour of different markets. Our strategy utilises two methods for detecting lead-lag relationships, with a method for computing *network momentum*, to produce a novel trend-following indicator. We use our new trend indicator to construct a portfolio whose performance we compare to a baseline model which uses only univariate indicators, and demonstrate statistically significant improvements in Sharpe ratio, skewness of returns, and downside performance, using synthetic bootstrapped data samples taken from time-series of actual prices.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In this phase, we define the configuration parameters, including the universe of tickers, strategy parameters, and our hypothesis.

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']  # Example universe, should be expanded to S&P 500 or subset
LOOKBACK_PERIOD = 20  # Period for calculating moving averages
LAG_PERIOD = 5  # Period for detecting lead-lag relationships
HYPOTHESIS = "The strategy will enhance trend-following performance by incorporating network momentum and lead-lag relationships."

## Phase 2 — Data Download & Feature Computation

In this phase, we download historical market data, compute necessary factors/features, and perform cross-sectional normalization.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download historical data
data = yf.download(UNIVERSE, start='2010-01-01', end='2023-01-01', group_by='ticker')['Adj Close']

# Compute moving averages
data_ma = data.rolling(window=LOOKBACK_PERIOD).mean()

# Compute lead-lag relationships
data_lag = data.shift(LAG_PERIOD)

# Cross-sectional normalization
data_normalized = (data - data.mean()) / data.std()

## Phase 3 — Signal Generation & Portfolio Construction

In this phase, we generate trading signals, size positions, and construct the portfolio.

In [ ]:
# Generate signals based on moving averages and lead-lag relationships
signals = (data_ma > data_lag).astype(int)

# Position sizing
positions = signals / signals.sum()

# Portfolio construction
portfolio = data.mul(positions).sum(axis=1)

## Phase 4 — Vectorized Backtest

In this phase, we perform a vectorized backtest of the strategy, ensuring no look-ahead bias by shifting signals forward by 1 period.

In [ ]:
# Shift signals forward by 1 period to avoid look-ahead bias
signals_shifted = signals.shift(1)

# Calculate daily returns
daily_returns = data.pct_change()

# Calculate strategy returns
strategy_returns = (daily_returns * signals_shifted).sum(axis=1)[1:]  # Skip the first row due to shifting

## Phase 5 — Performance Metrics

In this phase, we calculate performance metrics such as Sharpe ratio, Sortino ratio, Calmar ratio, max drawdown, and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Calculate performance metrics
cumulative_returns = (1 + strategy_returns).cumprod()
sharpe_ratio = np.sqrt(252) * strategy_returns.mean() / strategy_returns.std()
sortino_ratio = np.sqrt(252) * strategy_returns.mean() / strategy_returns[strategy_returns < 0].std()
max_drawdown = (cumulative_returns.cummax() - cumulative_returns).max()
calmar_ratio = sharpe_ratio / max_drawdown

# Plot equity curve
plt.figure(figsize=(10, 5))
plt.plot(cumulative_returns, label='Cumulative Returns')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

# Print performance metrics
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2f}')

## Phase 6 — Monitoring Stub

In this phase, we create a function that prints daily P&L and current positions given live data.

In [ ]:
# Monitoring function
def monitor_portfolio(live_data):
    live_signals = (live_data.rolling(window=LOOKBACK_PERIOD).mean() > live_data.shift(LAG_PERIOD)).astype(int)
    live_positions = live_signals / live_signals.sum()
    daily_pnl = (live_data.pct_change() * live_signals.shift(1)).sum(axis=1)
    print(f'Daily P&L: {daily_pnl[-1]:.2f}')
    print('Current Positions:')
    print(live_positions.iloc[-1])

# Example usage
live_data = yf.download(UNIVERSE, start='2023-01-01', end='2023-01-31', group_by='ticker')['Adj Close']
monitor_portfolio(live_data)